# Lab 9 - OOP II: Advanced Class Tools
**Difficulty: Intermediate | ~25 min | Requires Lab 8**

In [1]:
# This lab uses only Python's built-in features, but this cell keeps the
# notebook runnable from a fresh kernel. It installs nothing extra.
!pip install --upgrade pip

## Step 1 - What is an inner class?

In [2]:
# An inner (nested) class is simply a class defined inside another class.
# We usually do this when the inner concept only makes sense inside the outer one.

class Wrapper:
    # Child lives inside Wrapper -- you reach it as Wrapper.Child.
    class Child:
        def __init__(self, value):
            self.value = value

        def __str__(self):
            return f"Child({self.value})"

# To create one, we first name the outer class, then the inner class.
child = Wrapper.Child(42)
print(child)   # prints via Child.__str__

Child(42)


## Step 2 - Build the `Order` class in small pieces

### Step 2, part 1 - Inner `LineItem` class and order basics

In [3]:
class Order:
    # --- class attribute: state shared by every Order instance ---
    _id_counter = 0   # counts every Order we create (set from __init__)

    # --- inner class: one product line inside an order ---
    class LineItem:
        def __init__(self, name, quantity, price):
            # Validate at construction so an item can never be invalid.
            if quantity <= 0:
                raise ValueError("Quantity must be positive")
            if price <= 0:
                raise ValueError("Price must be positive")
            self.name = name        # product name, e.g. "Coffee"
            self.quantity = quantity
            self.price = price

        # Computed property: cost of this line (reads like a plain attribute).
        @property
        def total_price(self):
            return round(self.quantity * self.price, 2)

        # Human-readable line for print(order).
        def __str__(self):
            return f"  {self.name} x{self.quantity} @ ${self.price} = ${self.total_price}"

        # Developer-facing, unambiguous representation.
        def __repr__(self):
            return f"LineItem({self.name!r}, {self.quantity!r}, {self.price!r})"

        # Content equality: two lines are equal when all fields match.
        def __eq__(self, other):
            if not isinstance(other, Order.LineItem):
                return NotImplemented
            return (self.name == other.name
                    and self.quantity == other.quantity
                    and self.price == other.price)

        # len(item) returns how many units of this product were ordered.
        def __len__(self):
            return self.quantity

    # --- constructor ---
    def __init__(self, customer):
        Order._id_counter += 1   # bump the shared counter
        self.customer = customer # validated by the property below
        self.items = []          # empty list of lines; fill via add_item()

    def add_item(self, item):
        # One public way to grow the order's list of line items.
        self.items.append(item)

### Step 2, part 2 - Attach the static method

In [4]:
# A class is just an object, so we can attach more methods to Order
# from a new cell. staticmethod() wraps the raw function so it is not
# treated as an instance method.

@staticmethod
def calculate_discount(amount, percent):
    # Pure calculation; call it as Order.calculate_discount(100, 15).
    return round(amount * percent / 100, 2)

Order.calculate_discount = calculate_discount

### Step 2, part 3 - Attach the class methods

In [5]:
# classmethod() wraps the function in a descriptor that passes cls.

@classmethod
def from_dict(cls, data):
    # Builds an Order from a plain dictionary of items.
    order = cls(data["customer"])   # cls() constructs the right class
    for item in data["items"]:
        order.add_item(cls.LineItem(item["name"], item["quantity"], item["price"]))
    return order

Order.from_dict = from_dict

@classmethod
def get_order_count(cls):
    # Reads the class attribute shared by all instances.
    return cls._id_counter

Order.get_order_count = get_order_count

### Step 2, part 4 - Attach the properties

In [6]:
# property() combines the getter and setter into one object we assign.

@property
def total_price(self):
    # Read-only computed property: sum of every line's cost.
    return round(sum(item.total_price for item in self.items), 2)

Order.total_price = total_price

@property
def customer(self):
    return self._customer

@customer.setter
def customer(self, value):
    # Validation: an order must belong to a non-empty customer name.
    if not value or not value.strip():
        raise ValueError("Customer name cannot be empty")
    self._customer = value

Order.customer = customer

### Step 2, part 5 - Attach `__str__` and `__repr__`

In [7]:
def __str__(self):
    # Multi-line, human-readable receipt for print(order).
    lines = [f"Order for {self.customer}:"]
    for item in self.items:
        lines.append(str(item))
    lines.append(f"  Total: ${self.total_price}")
    return "\n".join(lines)

Order.__str__ = __str__

def __repr__(self):
    # Unambiguous developer representation.
    return f"Order(customer={self.customer!r}, items={self.items!r})"

Order.__repr__ = __repr__

### Step 2, part 6 - Attach `__eq__` and `__len__`

In [8]:
def __eq__(self, other):
    # Two orders are equal when customer and line items both match.
    if not isinstance(other, Order):
        return NotImplemented
    return self.customer == other.customer and self.items == other.items

Order.__eq__ = __eq__

def __len__(self):
    # len(order) = number of distinct line items.
    return len(self.items)

Order.__len__ = __len__

### Step 2, part 7 - Attach `__add__`

In [9]:
def __add__(self, other):
    # order1 + order2 creates a new Order that merges both line lists.
    if not isinstance(other, Order):
        return NotImplemented
    merged = Order(self.customer)                    # fresh order for the result
    for item in self.items:                          # copy this order's lines first
        merged.add_item(Order.LineItem(item.name, item.quantity, item.price))
    for other_item in other.items:                   # then the other order's lines
        merged_line = None
        for line in merged.items:
            if line.name == other_item.name:         # same product -> merge
                merged_line = line
                break
        if merged_line is not None:
            merged_line.quantity += other_item.quantity   # sum the quantities
        else:
            merged.add_item(Order.LineItem(
                other_item.name, other_item.quantity, other_item.price))
    return merged

Order.__add__ = __add__

## Step 3 - Static method demo

In [10]:
# Static methods need no instance: call them on the class directly.
print("Discount on $100 at 15%:", Order.calculate_discount(100, 15))
# A static method receives no self or cls, so it cannot touch object or
# class state -- it is just a tidy helper that lives on the class.

Discount on $100 at 15%: 15.0


## Step 4 - Class method demos

In [11]:
# from_dict is an alternative constructor driven by a dictionary.
order_a = Order.from_dict({
    "customer": "Carol",
    "items": [
        {"name": "Laptop", "quantity": 1, "price": 999.99},
        {"name": "Mouse", "quantity": 2, "price": 25.50},
    ],
})
print(order_a)

# get_order_count reads the class-level counter shared by all instances.
print("Orders created so far:", Order.get_order_count())

Order for Carol:
  Laptop x1 @ $999.99 = $999.99
  Mouse x2 @ $25.5 = $51.0
  Total: $1050.99
Orders created so far: 1


## Step 5 - Property demos

In [12]:
# Build an order manually, then read the computed total property.
order_b = Order("Carol")
order_b.add_item(Order.LineItem("Laptop", 1, 999.99))
order_b.add_item(Order.LineItem("Mouse", 2, 25.50))
print("Total via property:", order_b.total_price)   # computed on demand

# The customer setter validates its input before storing.
try:
    order_b.customer = None
except ValueError as error:
    print("Empty customer rejected:", error)

# LineItem validates its own values in the constructor.
try:
    Order.LineItem("Bad", 0, 5.0)
except ValueError as error:
    print("Line item validation:", error)

Total via property: 1050.99
Empty customer rejected: Customer name cannot be empty
Line item validation: Quantity must be positive


## Step 6 - `__str__` vs `__repr__` demo

In [13]:
# __str__ is the friendly text for print(); __repr__ is for developers.
order_c = Order("Dave")
order_c.add_item(Order.LineItem("Bread", 3, 3.0))
order_c.add_item(Order.LineItem("Tea", 1, 4.0))
print("str output:")
print(order_c)
print("repr output:")
print(repr(order_c))

str output:
Order for Dave:
  Bread x3 @ $3.0 = $9.0
  Tea x1 @ $4.0 = $4.0
  Total: $13.0
repr output:
Order(customer='Dave', items=[LineItem('Bread', 3, 3.0), LineItem('Tea', 1, 4.0)])


## Step 7 - `__eq__` and `__len__` demo

In [14]:
# Content equality and length.
order_d = Order("Eve")
order_d.add_item(Order.LineItem("Cake", 1, 20.0))
order_e = Order("Eve")
order_e.add_item(Order.LineItem("Cake", 1, 20.0))
print("order_d == order_e:", order_d == order_e)   # same customer + items
order_f = Order("Eve")
order_f.add_item(Order.LineItem("Juice", 2, 5.0))
print("order_d == order_f:", order_d == order_f)   # different items
print("len(order_d):", len(order_d))               # one line item

order_d == order_e: True
order_d == order_f: False
len(order_d): 1


## Step 8 - `__add__` demo

In [15]:
# Combine two orders with +, merging lines that share a product name.
order1 = Order("Frank")
order1.add_item(Order.LineItem("Apple", 3, 2.0))
order1.add_item(Order.LineItem("Banana", 2, 1.5))

order2 = Order("Frank")
order2.add_item(Order.LineItem("Apple", 1, 2.0))   # same product -> merges
order2.add_item(Order.LineItem("Cherry", 4, 0.75))

combined_demo = order1 + order2
print(combined_demo)

Order for Frank:
  Apple x4 @ $2.0 = $8.0
  Banana x2 @ $1.5 = $3.0
  Cherry x4 @ $0.75 = $3.0
  Total: $14.0


## Step 9 - Everything together

In [16]:
# Everything together: construct, print, compare, and combine.
order_a2 = Order("Alice")
order_a2.add_item(Order.LineItem("Coffee", 2, 5.0))
order_a2.add_item(Order.LineItem("Milk", 1, 8.0))
print(order_a2)

order_b2 = Order("Bob")
order_b2.add_item(Order.LineItem("Bread", 3, 3.0))
order_b2.add_item(Order.LineItem("Tea", 1, 4.0))
print(order_b2)

print("repr(order_a2):", repr(order_a2))
print("Orders created so far:", Order.get_order_count())

combined = order_a2 + order_b2
print("Combined order:")
print(combined)
print("len(combined):", len(combined))

Order for Alice:
  Coffee x2 @ $5.0 = $10.0
  Milk x1 @ $8.0 = $8.0
  Total: $18.0
Order for Bob:
  Bread x3 @ $3.0 = $9.0
  Tea x1 @ $4.0 = $4.0
  Total: $13.0
repr(order_a2): Order(customer='Alice', items=[LineItem('Coffee', 2, 5.0), LineItem('Milk', 1, 8.0)])
Orders created so far: 11
Combined order:
Order for Alice:
  Coffee x2 @ $5.0 = $10.0
  Milk x1 @ $8.0 = $8.0
  Bread x3 @ $3.0 = $9.0
  Tea x1 @ $4.0 = $4.0
  Total: $31.0
len(combined): 4


## Optional Exercise

Add a `remove_item(self, product_name)` method to `Order` that removes the first `LineItem` matching `product_name`. If the product is not found, raise a `ValueError` with the message `"Product not found in order"`.

Then:

1. Create an order with three line items.
2. Print the order and confirm `len()` is 3.
3. Remove the middle item.
4. Print the order again and confirm `len()` is 2 and the total has updated.

In [17]:
# Try the optional exercise here. A starting hint:
#
# def remove_item(self, product_name):
#     for i, item in enumerate(self.items):
#         if item.name == product_name:
#             self.items.pop(i)
#             return
#     raise ValueError("Product not found in order")
#
# Order.remove_item = remove_item